### <center>سوف نستخدم مكتبة Plotly الشهيرة مفتوحة المصدر لتصور البيانات.</center>



سترى هنا كيفية رسم الرسومات التفاعلية بسهولة وبأقل قدر ممكن من التعليمات البرمجية. تعد Plotly أداة قوية للغاية ومن المستحيل تغطية جميع ميزاتها مرة واحدة، لذا سأوضح لك كيفية إنشاء الرسومات الأكثر صلة وإثارة للاهتمام.



- <a href='#pie'>Pie</a>  
- <a href='#bar'>Bar</a>  
- <a href='#scatter'>Scatter</a> 
- <a href='#box'>Box</a>  
- <a href='#choropleth'>Choropleth</a>  



أولاً، إذا لم يكن Plotly مثبتًا لديك بالفعل، فقم بتشغيل:


In [ ]:
#!pip install plotly

In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.offline as py
import pycountry
import seaborn as sns
from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")

py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

In [ ]:
PATH = "120-years-of-olympic-history-athletes-and-results/athlete_events.csv"
data = pd.read_csv(PATH)
data.head(3)


لنقم بتنزيل الملف `athlete_events.csv` من [صفحة Kaggle](https://www.kaggle.com/heesoo37/120-years-of-olympic-history-athletes-and-results). تحتوي مجموعة البيانات على الميزات التالية:
- __ID__ - رقم فريد لكل رياضي
- __الاسم__ - اسم الرياضي
- __الجنس__ - M أو F
- __العمر__ - عدد صحيح
- __الارتفاع__ - بالسنتيمتر
- __الوزن__ - بالكيلو جرام
- __الفريق__ - اسم الفريق
- __NOC__ - رمز اللجنة الأولمبية الوطنية المكون من 3 أحرف
- __الألعاب__ - السنة والموسم
- __السنة__ - عدد صحيح
- __الموسم__ - الصيف أو الشتاء
- __المدينة__ - المدينة المضيفة
- __رياضة__ - رياضة
- __الحدث__ - الحدث
- __الميدالية__ - ذهبية، فضية، برونزية، أو غير متوفر



## <a id="pie">1. فطيرة</a>  



دعونا نرسم أبسط رسم بياني ممكن. عرض النسب المئوية لثلاثة أنواع من الأوسمة من إجمالي عدد الأوسمة.


In [ ]:
colors = ["#f4cb42", "#cd7f32", "#a1a8b5"]  # gold,bronze,silver
medal_counts = data.Medal.value_counts(sort=True)
labels = medal_counts.index
values = medal_counts.values

pie = go.Pie(labels=labels, values=values, marker=dict(colors=colors))
layout = go.Layout(title="Medal distribution")
fig = go.Figure(data=[pie], layout=layout)
py.iplot(fig)


توجد جميع الفئات الرئيسية لرسم الرسوم البيانية في <strong>plotly.graph_objs</strong> كـ <strong>go</strong>:- __go.Pie__ هو كائن رسم بياني يحتوي على أي من الوسائط أو السمات المذكورة أدناه. 
- __go.Layout__ يسمح لك بتخصيص تسميات المحاور والعناوين والخطوط والأحجام والهوامش والألوان والمزيد لتحديد مظهر المخطط.
- __go.Figer__ يقوم فقط بإنشاء الكائن النهائي المراد رسمه، وببساطة يقوم فقط بإنشاء كائن يشبه القاموس يحتوي على كل من كائن البيانات وكائن التخطيط.



حسنًا، كان ذلك سهلاً للغاية. دعونا نعقد الرسم البياني قليلاً، سنستخدم حلقتين على مخطط واحد.
سنعرض أفضل 10 دول فاز رياضيوها بأي ميداليات. منفصلة للرجال والنساء.


In [ ]:
topn = 10
male = data[data.Sex == "M"]
female = data[data.Sex == "F"]
count_male = male.dropna().NOC.value_counts()[:topn].reset_index()
count_female = female.dropna().NOC.value_counts()[:topn].reset_index()

pie_men = go.Pie(
    labels=count_male["index"],
    values=count_male.NOC,
    name="Men",
    hole=0.4,
    domain={"x": [0, 0.46]},
)
pie_women = go.Pie(
    labels=count_female["index"],
    values=count_female.NOC,
    name="Women",
    hole=0.4,
    domain={"x": [0.5, 1]},
)

layout = dict(
    title="Top-10 countries with medals by sex",
    font=dict(size=15),
    legend=dict(orientation="h"),
    annotations=[
        dict(x=0.2, y=0.5, text="Men", showarrow=False, font=dict(size=20)),
        dict(x=0.8, y=0.5, text="Women", showarrow=False, font=dict(size=20)),
    ],
)

fig = dict(data=[pie_men, pie_women], layout=layout)
py.iplot(fig)


- تحدد المعلمة __hole__ حجم الفتحة الموجودة في وسط الفطيرة
- تقوم المعلمة __domain__ بتعيين الإزاحة. تقوم المصفوفة X بتعيين الوضع الأفقي بينما تقوم المصفوفة Y بتعيين الوضع الرأسي. على سبيل المثال، x: [0,0.5]، y: [0, 0.5] يعني الموضع الأيسر السفلي من المخطط.
- يقوم الإملاء __annotations__ بتعيين تنسيق النص داخل الفطيرة.
- لمعرفة المزيد، اقرأ <a href='@@KEEP_00002@@>go.Pie وثائق</a>



## <a id="bar">2. شريط</a>  



بالطبع، لا يمكننا الاستغناء عن البار. 
دعونا نرسم مخططًا شريطيًا لعدد الألعاب الرياضية في سنوات مختلفة.


In [ ]:
games = data[data.Season == "Summer"].Games.unique()
games.sort()
sport_counts = np.array(
    [data[data.Games == game].groupby("Sport").size().shape[0] for game in games]
)
bar = go.Bar(
    x=games,
    y=sport_counts,
    marker=dict(color=sport_counts, colorscale="Reds", showscale=True),
)
layout = go.Layout(title="Number of sports in the summer Olympics by year")
fig = go.Figure(data=[bar], layout=layout)
py.iplot(fig)


نظام العرض بأكمله هو نفسه، الآن الفئة الأساسية هي __go.Bar__.
- يقوم القاموس __marker__ بتعيين نمط الرسم للمخطط ويسمح لك بعرض مقياس الألوان
- لمعرفة المزيد، اقرأ وثائق <a href='@@KEEP_00003@@>go.Bar</a>



مرة أخرى، دعونا نعقد الرسم البياني ونعرض عدد الميداليات المختلفة لأفضل 10 دول


In [ ]:
topn = 10
top10 = data.dropna().NOC.value_counts()[:topn]

gold = data[data.Medal == "Gold"].NOC.value_counts()
gold = gold[top10.index]
silver = data[data.Medal == "Silver"].NOC.value_counts()
silver = silver[top10.index]
bronze = data[data.Medal == "Bronze"].NOC.value_counts()
bronze = bronze[top10.index]

bar_gold = go.Bar(x=gold.index, y=gold, name="Gold", marker=dict(color="#f4cb42"))
bar_silver = go.Bar(
    x=silver.index, y=silver, name="Silver", marker=dict(color="#a1a8b5")
)
bar_bronze = go.Bar(
    x=bronze.index, y=bronze, name="Bronze", marker=dict(color="#cd7f32")
)

layout = go.Layout(
    title="Top-10 countries with medals", yaxis=dict(title="Count of medals")
)

fig = go.Figure(data=[bar_gold, bar_silver, bar_bronze], layout=layout)
py.iplot(fig)


## <a id="scatter">3. مبعثر</a>  



دعونا نرسم مخططًا مبعثرًا جميلًا يوضح متوسط الطول والوزن للرياضيين من مختلف الألعاب الرياضية.سنصنع دوائر بأحجام مختلفة اعتمادًا على شعبية الرياضة، ونتيجة لذلك، حجم عينة الرياضيين.


In [ ]:
tmp = data.groupby(["Sport"])["Height", "Weight"].agg("mean").dropna()
df1 = pd.DataFrame(tmp).reset_index()
tmp = data.groupby(["Sport"])["ID"].count()
df2 = pd.DataFrame(tmp).reset_index()
dataset = df1.merge(df2)  # DataFrame with columns 'Sport', 'Height', 'Weight', 'ID'

scatterplots = list()
for sport in dataset["Sport"]:
    df = dataset[dataset["Sport"] == sport]
    trace = go.Scatter(
        x=df["Height"],
        y=df["Weight"],
        name=sport,
        marker=dict(symbol="circle", sizemode="area", sizeref=10, size=df["ID"]),
    )
    scatterplots.append(trace)

layout = go.Layout(
    title="Mean height and weight by sport",
    xaxis=dict(title="Height, cm"),
    yaxis=dict(title="Weight, kg"),
    showlegend=True,
)

fig = dict(data=scatterplots, layout=layout)
py.iplot(fig)


لقد كانت جميلة، أليس كذلك؟ يمكننا بشكل تفاعلي إزالة الرياضة التي نهتم بها وتكبيرها وتحليل الرسوم البيانية بكل طريقة ممكنة.
- يقوم القاموس __marker__ مرة أخرى بتعريف طريقة عرض الرسم، ويحدد نوع الشكل (جرب، على سبيل المثال، مربع)، والأبعاد، والمزيد. الاحتمالات لا حصر لها تقريبا.
- لمعرفة المزيد، اقرأ <a href='@@KEEP_00004@@>go.Scatter الوثائق</a>



## <a id="box">4. صندوق</a>  



سنعرض إحصائيات عن أعمار الرجال والنساء المشاركين في الألعاب الأولمبية باستخدام Boxplot.


In [ ]:
men = data[data.Sex == "M"].Age
women = data[data.Sex == "F"].Age

box_m = go.Box(x=men, name="Male", fillcolor="navy")
box_w = go.Box(x=women, name="Female", fillcolor="lime")
layout = go.Layout(title="Age by sex")
fig = go.Figure(data=[box_m, box_w], layout=layout)
py.iplot(fig)


- يصف هذا الرسم البياني توزيع البيانات. يتوافق الخط العمودي المركزي مع الوسيط، وتتوافق حدود المستطيل مع الربعين الأول والثالث. النقاط تظهر القيم المتطرفة. وبالإضافة إلى ذلك، يمكنك رؤية الحد الأدنى والحد الأقصى للقيم.
- من برأيك هو __أصغر__ (10 سنوات) و__أكبر__ (97 عامًا) مشارك في الألعاب الأولمبية؟ العثور عليهم :)
- لمعرفة المزيد، اقرأ وثائق <a href='@@KEEP_00005@@>go.Box</a>



## <a id="choropleth">5. تشوروبليث</a> 



دعونا نحدد عدد المشاركين الذين أرسلتهم بلدان مختلفة خلال فترة الألعاب الأولمبية بأكملها.


In [ ]:
#!pip install pycountry

In [ ]:
def get_name(code):
    """
    Translate code to name of the country
    """
    try:
        name = pycountry.countries.get(alpha_3=code).name
    except:
        name = code
    return name


country_number = pd.DataFrame(data.NOC.value_counts())
country_number["country"] = country_number.index
country_number.columns = ["number", "country"]
country_number.reset_index().drop(columns=["index"], inplace=True)
country_number["country"] = country_number["country"].apply(lambda c: get_name(c))
country_number.head(3)

In [ ]:
worldmap = [
    dict(
        type="choropleth",
        locations=country_number["country"],
        locationmode="country names",
        z=country_number["number"],
        autocolorscale=True,
        reversescale=False,
        marker=dict(line=dict(color="rgb(180,180,180)", width=0.5)),
        colorbar=dict(autotick=False, title="Number of athletes"),
    )
]

layout = dict(
    title="The Nationality of Athletes",
    geo=dict(showframe=False, showcoastlines=True, projection=dict(type="Mercator")),
)

fig = dict(data=worldmap, layout=layout)
py.iplot(fig, validate=False)


- لمعرفة المزيد، اقرأ وثائق <a href='@@KEEP_00006@@>go.Box</a>



هذا كل شيء، لقد تعلمت كيفية عمل الحبكة وأتقنت الرسومات التفاعلية البسيطة والجميلة.



#### روابط مفيدة
- <a href='@@KEEP_00007@@>موقع بلوتلي</a>
- <a href='@@KEEP_00008@@>Plotly للمبتدئين</a>
- <a href='@@KEEP_00009@@> دروس مؤامرة</a>